# spaDIVA tutorial: human lymph node RNA-ADT data

This tutorial demonstrates a single-slice RNA-ADT analysis with spaDIVA, including modality-specific graph construction.


## 1. Import packages


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch
from sklearn.decomposition import PCA
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
from spaDIVA import build_modality_graphs, cal_spatial, cal_weight, cluster, clr_normalize_each_cell, infer_latents, lsi, train_spadiva

sc.set_figure_params(figsize=(3, 3))
plt.rcParams["figure.dpi"] = 120


## 2. Set random seed


In [ ]:
RANDOM_SEED = 42
POE_SAMPLE_SEED = RANDOM_SEED

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed()
USE_CUDA = False
USE_CUDA


## 3. Load data

Set `DATA_DIR` to the directory containing the human lymph node RNA-ADT files.

In [ ]:
DATA_ROOT = Path(os.environ.get("SPADIVA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
DATA_DIR = DATA_ROOT / "datasets" / "human_lymph_node_RNA_ADT" / "input_data"
ANNOTATION_DIR = DATA_ROOT / "datasets" / "human_lymph_node_RNA_ADT" / "annotation"
adata_omics1 = sc.read(DATA_DIR / "human lymph node adata_ADT.h5ad")
adata_omics2 = sc.read(DATA_DIR / "human lymph node adata_RNA.h5ad")
adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()
adata_omics1.X = adata_omics1.X.astype("float32")
adata_omics2.X = adata_omics2.X.astype("float32")


## 4. Load annotations


In [ ]:
Ann_df = pd.read_csv(ANNOTATION_DIR / "annotation.csv")
Ann_df = Ann_df.set_index("Barcode")
adata_omics1.obs["ground_truth"] = Ann_df["manual-anno"]
Y = adata_omics1.obs["ground_truth"]


## 5. Preprocess RNA and ADT modalities


In [ ]:
sc.pp.filter_genes(adata_omics2, min_cells=10)
sc.pp.highly_variable_genes(adata_omics2, flavor="seurat_v3", n_top_genes=3000)
sc.pp.normalize_total(adata_omics2, target_sum=1e4)
sc.pp.log1p(adata_omics2)
sc.pp.scale(adata_omics2)
adata_omics2 = adata_omics2[:, adata_omics2.var.highly_variable]
adata_omics1 = adata_omics1[adata_omics2.obs_names].copy()
adata_omics1 = clr_normalize_each_cell(adata_omics1)
sc.pp.scale(adata_omics1)
from sklearn.decomposition import  PCA
PREPROCESS_SEED = 0
set_seed(PREPROCESS_SEED)

pca1 = PCA(n_components=adata_omics1.n_vars - 1, svd_solver="full")
adata_omics1.obsm["clr"] = pca1.fit_transform(adata_omics1.to_df())

pca2 = PCA(
    n_components=adata_omics1.n_vars - 1,
    svd_solver="randomized",
    random_state=PREPROCESS_SEED,
)
adata_omics2.obsm["pca"] = pca2.fit_transform(adata_omics2.to_df())


## 6. Build spatial-feature graphs and training matrices


In [ ]:
spatial = adata_omics1.obsm["spatial"]
X1_train = adata_omics1.obsm["clr"]
X2_train = adata_omics2.obsm["pca"]
X1_input = adata_omics1.obsm["clr"]
X2_input = adata_omics2.obsm["pca"]
Y = adata_omics1.obs["ground_truth"].values

edge_index = cal_spatial(spatial, k=3)
edge_index_fused1, edge_weight_fused1, edge_index_fused2, edge_weight_fused2 = build_modality_graphs(
    X1_input,
    X2_input,
    edge_index,
    k_feature=20,
    spatial_weight=0.8,
    feature_weight=0.2,
    feature_metric="correlation",
)


## 7. Train spaDIVA once


In [ ]:
LEARNING_RATE = 1e-3
WEIGHT = 1.0
MAX_EPOCHS = 200

set_seed(RANDOM_SEED)
model, train_loss = train_spadiva(
    X1_input,
    X2_input,
    X1_train,
    X2_train,
    edge_index1=edge_index_fused1,
    edge_weight1=edge_weight_fused1,
    edge_index2=edge_index_fused2,
    edge_weight2=edge_weight_fused2,
    learning_rate=LEARNING_RATE,
    weight=WEIGHT,
    max_epochs=MAX_EPOCHS,
    KL_weight=[1, 1, 1, 1],
    use_cuda=USE_CUDA,
)


## 8. Inspect training loss


In [ ]:
plt.plot(train_loss)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("spaDIVA training loss")
plt.show()


## 9. Infer latent representations


In [ ]:
Z_poe, Z1_loc, Z2_loc, W1_loc, W2_loc, X1_hat, X2_hat = infer_latents(
    model,
    X1_input,
    X2_input,
    edge_index1=edge_index_fused1,
    edge_weight1=edge_weight_fused1,
    edge_index2=edge_index_fused2,
    edge_weight2=edge_weight_fused2,
    use_cuda=USE_CUDA,
    sample_seed=POE_SAMPLE_SEED,
)

a1, a2 = cal_weight(Z1_loc, Z2_loc, k=20)
Z_loc = a1.reshape(-1, 1) * Z1_loc + a2.reshape(-1, 1) * Z2_loc
Z_W = np.concatenate((W1_loc, Z_loc, W2_loc), axis=1)


## 10. Collect spaDIVA outputs


In [ ]:
z_adata = sc.AnnData(X=Z_loc)
z_adata.obsm["Z_W"] = Z_W
z_adata.obsm["Z"] = Z_loc
z_adata.obsm["Z_poe"] = Z_poe
z_adata.uns["preprocessing_seed"] = PREPROCESS_SEED
z_adata.uns["model_seed"] = RANDOM_SEED
z_adata.uns["poe_sample_seed"] = POE_SAMPLE_SEED
z_adata.obsm["W_ADT"] = W1_loc
z_adata.obsm["W_RNA"] = W2_loc
z_adata.obsm["spatial"] = spatial
z_adata.obs_names = adata_omics1.obs_names.copy()


## 11. Cluster and visualize representations

Plots are displayed inline.


In [ ]:
z_adata.obs["mclust_Z"] = cluster(Z_loc, num_cluster=6, spatial=spatial, title="Shared representation", s=30, show=True, return_labels=True)
z_adata.obs["mclust_Z_W"] = cluster(Z_W, num_cluster=6, spatial=spatial, title="Integrated representation", random_seed=RANDOM_SEED, s=30, show=True, return_labels=True)
z_adata.obs["mclust_W_ADT"] = cluster(W1_loc, num_cluster=6, spatial=spatial, title="ADT-specific representation", random_seed=RANDOM_SEED, s=30, show=True, return_labels=True)
z_adata.obs["mclust_W_RNA"] = cluster(W2_loc, num_cluster=6, spatial=spatial, title="RNA-specific representation", random_seed=RANDOM_SEED, s=30, show=True, return_labels=True)

metrics = pd.DataFrame([
    ("Z", adjusted_rand_score(Y, z_adata.obs["mclust_Z"]), normalized_mutual_info_score(Y, z_adata.obs["mclust_Z"])),
    ("Z_W", adjusted_rand_score(Y, z_adata.obs["mclust_Z_W"]), normalized_mutual_info_score(Y, z_adata.obs["mclust_Z_W"])),
    ("W_ADT", adjusted_rand_score(Y, z_adata.obs["mclust_W_ADT"]), normalized_mutual_info_score(Y, z_adata.obs["mclust_W_ADT"])),
    ("W_RNA", adjusted_rand_score(Y, z_adata.obs["mclust_W_RNA"]), normalized_mutual_info_score(Y, z_adata.obs["mclust_W_RNA"])),
], columns=["representation", "ARI", "NMI"])
metrics


## 12. Result object


In [ ]:
z_adata
